# Traduction automatique avec un transformeur

Dans ce TP, nous allons mettre en place un modèle de traduction de l'anglais vers le français :

1. utiliser un transformeur **encodeur-décodeur** pré-entraîné, [MarianMT](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr) ;
2. comprendre ce qui se passe pendant l'entraînement (*teacher forcing*) et la génération (recherche en faisceau) ;
3. mesurer la qualité des traductions avec le score BLEU, et ce que la recherche en faisceau rapporte pour ce qu'elle coûte ;
4. affiner (*fine-tuner*) le modèle sur un corpus de traductions de romans, avec une boucle d'entraînement PyTorch.

*Pensez à activer le GPU : Exécution > Modifier le type d'exécution > GPU.*

In [ ]:
!pip install -q datasets sacrebleu sentencepiece sacremoses

In [ ]:
import random
import time

import sacrebleu
import torch
import tqdm.auto
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")
torch.manual_seed(0)
random.seed(0)

## Chargement du modèle

Les modèles `opus-mt` de l'université d'Helsinki sont des transformeurs encodeur-décodeur entraînés sur les corpus parallèles [OPUS](https://opus.nlpl.eu/), un modèle par paire de langues.

In [ ]:
nom_modele = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForSeq2SeqLM.from_pretrained(nom_modele).to(device)

Inspectez le modèle (`print(modele.config)`, `print(modele)`) :

- combien de couches ont l'encodeur et le décodeur ?
- quelle est la dimension du modèle et le nombre de têtes d'attention ?
- où se trouve l'attention croisée (*cross-attention*) ? À quoi sert-elle ?
- quelle est la taille du vocabulaire ? Quel type de tokenisation est utilisé ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(modele.config)
print(modele.model.decoder.layers[0])
print(sum(p.numel() for p in modele.parameters()) / 1e6, "millions de paramètres")
print(tokenizer.tokenize("The neighbour's cat is extraordinarily lazy."))

- L'encodeur et le décodeur ont chacun 6 couches (`encoder_layers`, `decoder_layers`), de dimension 512 (`d_model`) avec 8 têtes d'attention.
- Chaque couche du décodeur contient une auto-attention **causale** (`self_attn`) puis une attention croisée (`encoder_attn`) : les requêtes viennent du décodeur, les clés et valeurs de la sortie de l'encodeur. C'est ainsi que le décodeur « lit » la phrase source à chaque mot généré.
- Le vocabulaire compte environ 60 000 sous-mots, obtenus avec SentencePiece (le `▁` marque un début de mot).

## Traduire

La méthode `generate` implémente la boucle de décodage. Par défaut, ce modèle utilise une **recherche en faisceau** (*beam search*) : au lieu de ne garder que le mot le plus probable à chaque étape (décodage glouton), elle garde les `num_beams` débuts de traduction les plus probables et choisit à la fin la meilleure traduction complète.

In [ ]:
def traduit(
  phrases: list[str], nb_faisceaux: int = 4, taille_lot: int = 32
) -> list[str]:
  modele.eval()
  traductions = []
  for i in range(0, len(phrases), taille_lot):
    batch = tokenizer(
      phrases[i : i + taille_lot],
      return_tensors="pt",
      padding=True,
      truncation=True,
      max_length=128,
    ).to(device)
    with torch.no_grad():
      ids_generes = modele.generate(**batch, num_beams=nb_faisceaux, max_length=256)
    traductions += tokenizer.batch_decode(ids_generes, skip_special_tokens=True)
  return traductions


exemples = [
  "I think, therefore I am.",
  "The weather is lovely today, let's go for a walk.",
  "Could you please send me the report before Friday?",
]
for source, cible in zip(exemples, traduit(exemples)):
  print(f"{source}\n  → {cible}")

## Les données : traductions de romans

Nous utilisons [OPUS Books](https://huggingface.co/datasets/Helsinki-NLP/opus_books) : des phrases de romans du domaine public alignées avec leur traduction française. Le style littéraire (dialogues, tournures anciennes) est différent des textes sur lesquels le modèle a surtout été entraîné.

OPUS Books fait pourtant partie d'OPUS, et OPUS-MT entraîne chaque modèle sur presque tous les corpus d'OPUS de sa paire de langues : nos phrases de test ont pu être vues pendant le pré-entraînement, ce qui flatterait le score du modèle avant affinage.

Environ une phrase française sur cinq de ce corpus utilise l'apostrophe typographique (`’`), absente du vocabulaire du modèle : elle deviendrait `<unk>`, que `skip_special_tokens=True` efface au décodage, et un modèle affiné écrirait « simposait » pour « s’imposait ». Le score BLEU, lui, compterait comme une erreur chaque apostrophe droite du modèle face à une apostrophe typographique de la référence. Nous la remplaçons donc par l'apostrophe droite dès la lecture des données.

Nous gardons 1 000 paires pour le test, 1 000 pour la validation (le contrôle du surapprentissage pendant l'affinage) et 20 000 pour l'entraînement (pour que le TP reste rapide).

In [ ]:
livres = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")


def redresse_apostrophes(exemple: dict) -> dict:
  # Le vocabulaire du modèle n'a que l'apostrophe droite
  paire = exemple["translation"]
  return {"translation": {k: texte.replace("’", "'") for k, texte in paire.items()}}


livres = livres.map(redresse_apostrophes).shuffle(seed=0)
paires_test = livres.select(range(1000))
paires_val = livres.select(range(1000, 2000))
paires_ent = livres.select(range(2000, 22000))
for paire in paires_test.select(range(3))["translation"]:
  print(paire)

## Mesurer la qualité : le score BLEU

Le score [BLEU](https://fr.wikipedia.org/wiki/BLEU_(algorithme)) compare les *n*-grammes (suites de 1 à 4 mots) d'une traduction automatique à ceux d'une traduction de référence. Il va de 0 à 100. Les seuils souvent cités (au-delà de 30, des traductions compréhensibles ; au-delà de 50, de bonne qualité) ne sont qu'un ordre de grandeur : le score dépend de la langue, du corpus, de la tokenisation et de la liberté des traductions de référence. Les traducteurs de romans en prennent beaucoup : des traductions correctes y obtiennent un score plus bas que sur des textes techniques.

C'est une mesure imparfaite (une traduction correcte mais formulée autrement est pénalisée), mais elle permet de comparer des modèles sur un même corpus. chrF (`sacrebleu.corpus_chrf`), qui compte les *n*-grammes de caractères, tolère mieux les variations de forme ; COMET, un modèle appris, juge plus près d'un humain.

Codez une fonction `bleu(paires, nb_faisceaux)` qui traduit les phrases anglaises de `paires` avec `traduit` et renvoie le score BLEU de ces traductions, calculé avec `sacrebleu.corpus_bleu`. Calculez `bleu_avant`, le score du modèle pré-entraîné sur l'ensemble de test.

In [ ]:
def bleu(paires, nb_faisceaux: int = 4) -> float:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `paires["translation"]` : la liste des paires, chacune un dictionnaire `{"en": ..., "fr": ...}`
- `traduit(sources, nb_faisceaux=nb_faisceaux)` : les traductions, dans l'ordre des sources
- `sacrebleu.corpus_bleu(traductions, [références])` : la liste des traductions, puis une liste de séries de références (une série par traducteur, ici une seule) ; le score est dans l'attribut `score`
- Le modèle pré-entraîné obtient environ 23 sur ce test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def bleu(paires, nb_faisceaux: int = 4) -> float:
  sources = [paire["en"] for paire in paires["translation"]]
  references = [paire[...] for paire in ...]
  return sacrebleu.corpus_bleu(traduit(..., nb_faisceaux=...), [...]).score


bleu_avant = bleu(...)
print(f"BLEU avant fine-tuning : {bleu_avant:.1f}")
```

### Solution

In [ ]:
def bleu(paires, nb_faisceaux: int = 4) -> float:
  sources = [paire["en"] for paire in paires["translation"]]
  references = [paire["fr"] for paire in paires["translation"]]
  return sacrebleu.corpus_bleu(
    traduit(sources, nb_faisceaux=nb_faisceaux), [references]
  ).score


bleu_avant = bleu(paires_test)
print(f"BLEU avant fine-tuning : {bleu_avant:.1f}")

## Glouton ou faisceau ?

La recherche en faisceau trouve des traductions plus probables que le décodage glouton, mais elle suit `nb_faisceaux` débuts de traduction à la fois. Gagne-t-elle assez en qualité pour ce qu'elle coûte ?

Mesurez le score BLEU et le temps de traduction des 200 premières phrases de test (`paires_test.select(range(200))`) avec `nb_faisceaux=1` (décodage glouton) et `nb_faisceaux=4`, en chronométrant chaque appel de `bleu` avec `time.perf_counter`. Concluez.

**Documentation utile** :

- https://docs.python.org/3/library/time.html#time.perf_counter

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sous_ensemble = paires_test.select(range(200))` : les 200 premières paires, un `Dataset` comme `paires_test`
- `debut = time.perf_counter()` avant l'appel, `time.perf_counter() - debut` après : la durée en secondes
- `bleu(sous_ensemble, nb_faisceaux=nb_faisceaux)` dans une boucle sur `(1, 4)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
sous_ensemble = paires_test.select(range(200))
for nb_faisceaux in (1, 4):
  debut = time.perf_counter()
  score = bleu(..., nb_faisceaux=...)
  duree = ... - debut
  print(f"nb_faisceaux={nb_faisceaux} : BLEU {score:.1f} en {duree:.1f} s")
```

### Solution

In [ ]:
sous_ensemble = paires_test.select(range(200))
for nb_faisceaux in (1, 4):
  debut = time.perf_counter()
  score = bleu(sous_ensemble, nb_faisceaux=nb_faisceaux)
  duree = time.perf_counter() - debut
  print(f"nb_faisceaux={nb_faisceaux} : BLEU {score:.1f} en {duree:.1f} s")

Sur ces 200 phrases, le faisceau gagne environ un point de BLEU (23,7 contre 22,8 pour le décodage glouton), et le temps de traduction double à peu près sur GPU (7,2 s contre 3,6 s), contre environ 60 % de plus sur CPU ; l'écart dépend de la taille des batchs. La recherche en faisceau n'est qu'une recherche approchée de la traduction la plus probable, plus large que le glouton : elle en trouve de meilleures, mais le gain est modeste. Le choix dépend de l'usage : le glouton pour traduire beaucoup de texte vite, le faisceau quand chaque traduction compte (c'est le réglage par défaut de ce modèle, `num_beams=4`).

## Sous le capot : le *teacher forcing*

Pendant l'entraînement, on ne génère pas mot à mot : on donne au décodeur la **traduction de référence** décalée d'un cran, et on lui demande de prédire chaque mot suivant. Grâce au masque causal, toutes les positions sont prédites en une seule passe.

Le tokeniseur prépare les deux côtés : `text_target` produit les `labels`, et le modèle calcule lui-même la perte d'entropie croisée quand on lui donne ces `labels`.

Calculez la perte du modèle sur la paire de phrases de la cellule suivante, sans calculer de gradient. Affichez la forme des `logits` : à quoi correspond chaque dimension ?

**Documentation utile** :

- https://huggingface.co/docs/transformers/main_classes/output#transformers.modeling_outputs.Seq2SeqLMOutput

In [ ]:
paire = paires_ent[0]["translation"]
batch = tokenizer(paire["en"], text_target=paire["fr"], return_tensors="pt").to(device)
print(batch.keys())
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.no_grad()` : pas de gradient pour une simple mesure
- `sortie = modele(**batch)` : avec les `labels` du batch, `sortie.loss` est l'entropie croisée moyenne sur les tokens de la traduction
- `sortie.logits.shape` : trois dimensions, dont la dernière vaut 59514
- `tokenizer.convert_ids_to_tokens(batch["labels"][0])` : les sous-mots de la traduction de référence, pour compter les positions

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
with torch.no_grad():
  sortie = modele(...)
print(f"Perte : {sortie.loss.item():.3f}")
print(f"Forme des logits : {tuple(...)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))
```

### Solution

In [ ]:
with torch.no_grad():
  sortie = modele(**batch)
print(f"Perte : {sortie.loss.item():.3f}")
print(f"Forme des logits : {tuple(sortie.logits.shape)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))

Les logits ont la forme `(batch, longueur de la traduction, taille du vocabulaire)` : pour chaque position de la traduction de référence, un score par sous-mot du vocabulaire.

## Préparation des données d'entraînement

Codez une fonction `pretraite(exemples)` qui tokenise un batch d'exemples du dataset (sources anglaises et cibles françaises, tronquées à 128 tokens) avec `tokenizer(..., text_target=...)`, puis appliquez-la à `paires_ent` avec `Dataset.map` (`batched=True`) pour obtenir `ent_tokenise`, sans garder la colonne de texte d'origine.

Le `DataCollatorForSeq2Seq` fourni ensuite complète les séquences d'un batch avec du padding. Pourquoi remplace-t-il le padding des `labels` par `-100` ?

**Documentation utile** :

- https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.map

In [ ]:
def pretraite(exemples: dict) -> dict:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `exemples["translation"]` : avec `batched=True`, la liste des paires d'un batch, chacune `{"en": ..., "fr": ...}`
- `tokenizer(sources, text_target=cibles, max_length=128, truncation=True)` : `input_ids` et `attention_mask` des sources, `labels` des cibles ; pas de padding ici, le collator le fait batch par batch
- `remove_columns=paires_ent.column_names` : retire la colonne `translation`, du texte que le collator ne saurait pas assembler en tenseurs
- `ent_tokenise` a 20 000 lignes et les colonnes `input_ids`, `attention_mask` et `labels`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def pretraite(exemples: dict) -> dict:
  sources = [paire["en"] for paire in exemples[...]]
  cibles = [... for paire in exemples["translation"]]
  return tokenizer(sources, text_target=..., max_length=..., truncation=True)


ent_tokenise = paires_ent.map(pretraite, batched=True, remove_columns=...)
print(ent_tokenise)
```

### Solution

In [ ]:
def pretraite(exemples: dict) -> dict:
  sources = [paire["en"] for paire in exemples["translation"]]
  cibles = [paire["fr"] for paire in exemples["translation"]]
  return tokenizer(sources, text_target=cibles, max_length=128, truncation=True)


ent_tokenise = paires_ent.map(
  pretraite, batched=True, remove_columns=paires_ent.column_names
)
print(ent_tokenise)

`-100` est l'indice que `cross_entropy` ignore (`ignore_index`) : les positions de padding ne comptent pas dans la perte.

### Les batchs

Le collator assemble les batchs de `chargeur_ent`, et ceux de `chargeur_val`, qui mesurera la perte de validation à la fin de l'affinage.

In [ ]:
assembleur = DataCollatorForSeq2Seq(tokenizer, model=modele)
chargeur_ent = DataLoader(
  ent_tokenise, batch_size=32, shuffle=True, collate_fn=assembleur
)
val_tokenise = paires_val.map(
  pretraite, batched=True, remove_columns=paires_val.column_names
)
chargeur_val = DataLoader(val_tokenise, batch_size=64, collate_fn=assembleur)
batch = next(iter(chargeur_ent))
print({cle: valeur.shape for cle, valeur in batch.items()})

## Fine-tuning

Avant d'affiner le modèle, gardons ses traductions de cinq phrases de test : la dernière partie les comparera à celles du modèle affiné.

In [ ]:
echantillons = paires_test.select(range(5))["translation"]
avant = traduit([paire["en"] for paire in echantillons])

Codez la boucle d'entraînement pour une epoch :

- optimiseur `AdamW`, learning rate de `5e-5` (petit : on ajuste un modèle déjà bon) ;
- un planificateur de learning rate linéaire avec *warmup* (`get_linear_schedule_with_warmup`, 10 % des pas en warmup) ;
- la perte est `modele(**batch).loss`.

À la fin de l'epoch, calculez la perte moyenne sur `chargeur_val`, en mode évaluation et sans gradient. Celle du modèle pré-entraîné est d'environ 2,1 : l'affinage la fait-il baisser ? Un écart qui se creuse avec la perte d'entraînement signalerait un surapprentissage.

Sur un GPU de Colab, une epoch prend quelques minutes.

**Documentation utile** :

- https://huggingface.co/docs/transformers/main_classes/optimizer_schedules#transformers.get_linear_schedule_with_warmup

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.optim.AdamW(modele.parameters(), lr=5e-5)`
- `nb_pas = len(chargeur_ent)` : le nombre de pas d'une epoch ; `get_linear_schedule_with_warmup(optimiseur, num_warmup_steps=nb_pas // 10, num_training_steps=nb_pas)`
- `planificateur.step()` après chaque `optimiseur.step()` : le planificateur avance d'un pas par batch, pas par epoch
- `{cle: valeur.to(device) for cle, valeur in batch.items()}` : le batch sur le GPU
- `modele.train()` avant la boucle (`traduit` a mis le modèle en mode évaluation), `modele.eval()` et `torch.no_grad()` pour la validation
- la perte de validation : la moyenne des `modele(**batch).loss.item()` sur les batchs de `chargeur_val`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from transformers import get_linear_schedule_with_warmup

optimiseur = torch.optim.AdamW(modele.parameters(), lr=...)
nb_pas = len(chargeur_ent)
planificateur = get_linear_schedule_with_warmup(
  optimiseur, num_warmup_steps=..., num_training_steps=...
)

modele.train()
perte_cumulee = 0.0
for pas, batch in enumerate(tqdm.auto.tqdm(chargeur_ent), 1):
  batch = {cle: valeur.to(device) for cle, valeur in batch.items()}
  optimiseur.zero_grad()
  perte = modele(**batch).loss
  ...
  optimiseur.step()
  ...
  perte_cumulee += perte.item()
  if pas % 100 == 0:
    print(f"Pas {pas} : perte {perte_cumulee / 100:.3f}")
    perte_cumulee = 0.0

modele.eval()
perte_val = 0.0
with torch.no_grad():
  for batch in chargeur_val:
    batch = {cle: valeur.to(device) for cle, valeur in batch.items()}
    perte_val += ...
print(f"Perte de validation : {perte_val / len(chargeur_val):.3f}")
```

### Solution

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimiseur = torch.optim.AdamW(modele.parameters(), lr=5e-5)
nb_pas = len(chargeur_ent)
planificateur = get_linear_schedule_with_warmup(
  optimiseur, num_warmup_steps=nb_pas // 10, num_training_steps=nb_pas
)

modele.train()
perte_cumulee = 0.0
for pas, batch in enumerate(tqdm.auto.tqdm(chargeur_ent), 1):
  batch = {cle: valeur.to(device) for cle, valeur in batch.items()}
  optimiseur.zero_grad()
  perte = modele(**batch).loss
  perte.backward()
  optimiseur.step()
  planificateur.step()
  perte_cumulee += perte.item()
  if pas % 100 == 0:
    print(f"Pas {pas} : perte {perte_cumulee / 100:.3f}")
    perte_cumulee = 0.0

modele.eval()
perte_val = 0.0
with torch.no_grad():
  for batch in chargeur_val:
    batch = {cle: valeur.to(device) for cle, valeur in batch.items()}
    perte_val += modele(**batch).loss.item()
print(f"Perte de validation : {perte_val / len(chargeur_val):.3f}")

## Évaluation après fine-tuning

Calculez `bleu_apres`, le score BLEU du modèle affiné sur l'ensemble de test, et comparez-le à `bleu_avant`. Traduisez à nouveau les phrases de `echantillons`, et affichez pour chacune la source, la référence, la traduction d'avant l'affinage (`avant`) et celle d'après. Que constatez-vous ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `bleu(paires_test)` : le même calcul qu'avant l'affinage
- `apres = traduit([paire["en"] for paire in echantillons])` : les nouvelles traductions, dans l'ordre de `avant`
- `zip(echantillons, avant, apres)` : chaque paire, sa traduction d'avant et celle d'après
- Avant l'affinage, le score était d'environ 23

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bleu_apres = bleu(...)
print(f"BLEU avant : {bleu_avant:.1f}, après : {bleu_apres:.1f}")

apres = traduit([paire["en"] for paire in ...])
for paire, ancien, nouveau in zip(echantillons, avant, apres):
  print(f"Source    : {paire['en']}")
  print(f"Référence : {paire['fr']}")
  print(f"Avant     : {...}")
  print(f"Après     : {...}")
  print("-" * 80)
```

### Solution

In [ ]:
bleu_apres = bleu(paires_test)
print(f"BLEU avant : {bleu_avant:.1f}, après : {bleu_apres:.1f}")

apres = traduit([paire["en"] for paire in echantillons])
for paire, ancien, nouveau in zip(echantillons, avant, apres):
  print(f"Source    : {paire['en']}")
  print(f"Référence : {paire['fr']}")
  print(f"Avant     : {ancien}")
  print(f"Après     : {nouveau}")
  print("-" * 80)

Le score BLEU progresse en général dès la première epoch (de 22,8 à 26,8 dans notre exécution), et la perte de validation baisse : le modèle s'adapte au style et au vocabulaire des romans. Les traductions affinées reprennent aussi davantage les tournures de la référence, sans être toutes meilleures : dans notre exécution, le « journeyman » d'un des exemples, « compagnon » avant l'affinage comme dans la référence, devient « voyageur ». C'est le principe du transfert d'apprentissage appliqué au texte : partir d'un modèle généraliste et l'ajuster avec peu de données du domaine visé.

Pour aller plus loin :

- entraînez plusieurs epochs en mesurant la perte de validation à la fin de chacune : quand elle cesse de baisser alors que la perte d'entraînement baisse encore, le modèle sur-apprend ;
- mesurez aussi chrF (`sacrebleu.corpus_chrf`) avant et après l'affinage : les deux mesures montrent-elles le même progrès ?
- sauvegardez le modèle affiné avec `modele.save_pretrained("opus-mt-en-fr-books")` et `tokenizer.save_pretrained(...)`, puis rechargez-le avec `from_pretrained`.